In [3]:

from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


class Transform:

    # ============================================================
    # CONFIGURAÇÃO
    # ============================================================

    def __init__(
        self,
        pasta_csv=r"dados\raw\cliente\csv",
        pasta_parquet=r"dados\staging\parquet"
    ):
        self.pasta_csv = Path(pasta_csv)
        self.pasta_parquet = Path(pasta_parquet)

        self.total_arquivos = 0
        self.total_processados = 0
        self.total_erros = 0

        self.arquivos_processados = []
        self.arquivos_erro = []

        self.pasta_parquet.mkdir(
            parents=True,
            exist_ok=True
        )

    # ============================================================
    # MÉTODO ESTÁTICO - NORMALIZAR DATAFRAME
    # ============================================================

    @staticmethod
    def normalizar_dataframe(df):

        df = df.copy().fillna("")

        for coluna in df.columns:
            df[coluna] = (
                df[coluna]
                .astype(str)
                .str.strip()
            )

        df.columns = [
            str(coluna).strip()
            for coluna in df.columns
        ]

        return df.loc[
            :,
            ~df.columns.duplicated()
        ]

    # ============================================================
    # MÉTODO ESTÁTICO - SALVAR PARQUET
    # ============================================================

    @staticmethod
    def salvar_parquet_string(df, caminho_saida):

        df = Transform.normalizar_dataframe(df)

        schema = pa.schema([
            pa.field(coluna, pa.string())
            for coluna in df.columns
        ])

        tabela = pa.Table.from_pandas(
            df,
            schema=schema,
            preserve_index=False
        )

        pq.write_table(
            tabela,
            caminho_saida,
            compression="snappy"
        )

        return tabela

    # ============================================================
    # MÉTODO ESTÁTICO - LOCALIZAR HEADER
    # ============================================================

    @staticmethod
    def localizar_header(caminho_arquivo):

        with open(
            caminho_arquivo,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as arquivo:

            for i, linha in enumerate(arquivo):

                linha = linha.lower()

                if "data" in linha and "lat" in linha:
                    return i

        return None

    # ============================================================
    # MÉTODO ESTÁTICO - EXTRAIR METADADOS
    # ============================================================

    @staticmethod
    def extrair_metadados(caminho_arquivo, linha_header):

        df = pd.read_csv(
            caminho_arquivo,
            nrows=linha_header,
            header=None,
            encoding="utf-8",
            engine="python"
        )

        df = df.dropna(axis=1, how="all")
        df = df.dropna(axis=0, how="all")
        df = df.fillna("")

        for coluna in df.columns:
            df[coluna] = (
                df[coluna]
                .astype(str)
                .str.strip()
            )

        df.columns = [
            f"coluna_{i}"
            for i in range(len(df.columns))
        ]

        df = (
            df.set_index("coluna_0")
            .T
            .reset_index(drop=True)
        )

        df.columns = [
            str(coluna).strip()
            for coluna in df.columns
        ]

        df = df.loc[
            :,
            ~df.columns.duplicated()
        ]

        return df

    # ============================================================
    # MÉTODO ESTÁTICO - EXTRAIR DADOS
    # ============================================================

    @staticmethod
    def extrair_dados(caminho_arquivo, linha_header):

        df = pd.read_csv(
            caminho_arquivo,
            skiprows=linha_header,
            encoding="utf-8",
            engine="python"
        )

        df = df.dropna(
            axis=1,
            how="all"
        )

        df.columns = [
            str(coluna).strip()
            for coluna in df.columns
        ]

        return df

    # ============================================================
    # MÉTODO ESTÁTICO - MOSTRAR SCHEMA
    # ============================================================

    @staticmethod
    def mostrar_schema(tabela, nome):

        print(f"\nSchema {nome}:")

        for campo in tabela.schema:
            print(
                f"    {campo.name}: {campo.type}"
            )

    # ============================================================
    # MÉTODO DE INSTÂNCIA - LOCALIZAR CSV
    # ============================================================

    def localizar_csv(self):

        if not self.pasta_csv.exists():
            raise FileNotFoundError(
                f"Pasta de entrada não encontrada: "
                f"{self.pasta_csv}"
            )

        arquivos = sorted(
            arquivo
            for arquivo in self.pasta_csv.iterdir()
            if arquivo.is_file()
            and arquivo.suffix.lower() == ".csv"
        )

        self.total_arquivos = len(arquivos)

        return arquivos

    # ============================================================
    # MÉTODO DE INSTÂNCIA - PROCESSAR ARQUIVO
    # ============================================================

    def processar_arquivo(self, arquivo):

        nome_base = arquivo.stem

        print("\n" + "=" * 70)
        print(f"PROCESSANDO: {arquivo.name}")
        print("=" * 70)

        linha_header = self.localizar_header(arquivo)

        if linha_header is None:
            raise ValueError(
                "Cabeçalho contendo 'data' e 'lat' "
                "não encontrado."
            )

        print(
            f"Header encontrado na linha: {linha_header}"
        )

        # --------------------------------------------------------
        # 1. EXTRAIR METADADOS
        # --------------------------------------------------------

        df_meta = self.extrair_metadados(
            arquivo,
            linha_header
        )

        caminho_meta = (
            self.pasta_parquet
            / f"{nome_base}_metadados.parquet"
        )

        tabela_meta = self.salvar_parquet_string(
            df_meta,
            caminho_meta
        )

        print(f"Metadados salvos: {caminho_meta}")

        # --------------------------------------------------------
        # 2. EXTRAIR DADOS
        # --------------------------------------------------------

        df_dados = self.extrair_dados(
            arquivo,
            linha_header
        )

        caminho_dados = (
            self.pasta_parquet
            / f"{nome_base}_dados.parquet"
        )

        tabela_dados = self.salvar_parquet_string(
            df_dados,
            caminho_dados
        )

        print(f"Dados salvos: {caminho_dados}")

        # --------------------------------------------------------
        # 3. EXIBIR SCHEMAS
        # --------------------------------------------------------

        self.mostrar_schema(
            tabela_meta,
            "METADADOS"
        )

        self.mostrar_schema(
            tabela_dados,
            "DADOS"
        )

        # --------------------------------------------------------
        # 4. REGISTRAR RESULTADO
        # --------------------------------------------------------

        self.total_processados += 1

        self.arquivos_processados.append({
            "arquivo": arquivo.name,
            "metadados": caminho_meta.name,
            "dados": caminho_dados.name
        })

        print("\nPROCESSAMENTO CONCLUÍDO")

    # ============================================================
    # MÉTODO DE INSTÂNCIA - EXECUTAR PIPELINE
    # ============================================================

    def executar(self):

        arquivos = self.localizar_csv()

        print("=" * 70)
        print("ETAPA 1 - LOCALIZAÇÃO DOS CSV")
        print("=" * 70)

        print(
            f"Total de arquivos encontrados: "
            f"{self.total_arquivos}"
        )

        print("\n")
        print("=" * 70)
        print("ETAPA 2 - TRANSFORMAÇÃO E GRAVAÇÃO PARQUET")
        print("=" * 70)

        for arquivo in arquivos:

            try:
                self.processar_arquivo(arquivo)

            except Exception as erro:

                self.total_erros += 1

                self.arquivos_erro.append({
                    "arquivo": arquivo.name,
                    "erro": str(erro)
                })

                print(
                    f"ERRO | {arquivo.name} | {erro}"
                )

        self.relatorio_final()

    # ============================================================
    # MÉTODO DE INSTÂNCIA - RELATÓRIO FINAL
    # ============================================================

    def relatorio_final(self):

        print("\n")
        print("=" * 70)
        print("RELATÓRIO FINAL")
        print("=" * 70)

        print(f"CSV encontrados : {self.total_arquivos}")
        print(f"Processados     : {self.total_processados}")
        print(f"Erros           : {self.total_erros}")
        print(f"Pasta de saída  : {self.pasta_parquet}")

        print("\nARQUIVOS PROCESSADOS")

        for item in self.arquivos_processados:
            print(f"CSV: {item['arquivo']}")
            print(f"    Metadados: {item['metadados']}")
            print(f"    Dados: {item['dados']}")

        if self.arquivos_erro:

            print("\nERROS")

            for item in self.arquivos_erro:
                print(f"Arquivo: {item['arquivo']}")
                print(f"Erro: {item['erro']}\n")

        else:
            print("\nNenhum erro encontrado.")

        print("=" * 70)
        print("PROCESSAMENTO FINALIZADO")


# ============================================================
# EXECUÇÃO
# ============================================================

if __name__ == "__main__":

    transform = Transform(
        pasta_csv=r"dados\raw\cliente\csv",
        pasta_parquet=r"dados\staging\parquet"
    )

    transform.executar()


ETAPA 1 - LOCALIZAÇÃO DOS CSV
Total de arquivos encontrados: 4


ETAPA 2 - TRANSFORMAÇÃO E GRAVAÇÃO PARQUET

PROCESSANDO: CLIENTE1_relatorio1_dados.csv
Header encontrado na linha: 24
Metadados salvos: dados\staging\parquet\CLIENTE1_relatorio1_dados_metadados.parquet
Dados salvos: dados\staging\parquet\CLIENTE1_relatorio1_dados_dados.parquet

Schema METADADOS:
    nome_arquivo: string
    coluna2: string
    coluna3: string
    coluna4: string
    coluna5: string
    coluna6: string
    coluna7: string
    coluna8: string
    coluna9: string
    coluna10: string
    coluna11: string
    coluna12: string
    coluna13: string
    coluna14: string
    coluna15: string
    coluna16: string
    coluna17: string
    coluna18: string
    coluna19: string
    coluna21: string
    coluna22: string
    coluna23: string
    coluna24: string
    coluna25: string

Schema DADOS:
    data: string
    speed: string
    yaw: string
    lat: string
    long: string
    bouncing: string

PROCESSAMENTO CON